In [1]:
%pip -q install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00


In [2]:
import gc
import json
import copy
import random

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import (resnet50, vit_b_16, ResNet50_Weights, ViT_B_16_Weights)

from torch.utils.data import (Dataset, DataLoader, TensorDataset)

from PIL import Image
from tqdm.auto import tqdm

import open_clip

SEED = 6304

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
PyTorch: 2.11.0+cu128
TorchVision: 0.26.0+cu128
GPU: Tesla T4


In [3]:

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/pa1-beyond-iid")

TASK_ROOT = PROJECT_ROOT / "task1"

DATA_PATH = TASK_ROOT / "data/stl10_labeled.npz"
SPLIT_PATH = TASK_ROOT / "data/split_indices.json"

FEATURE_DIR = TASK_ROOT / "results/features"
HEAD_DIR = TASK_ROOT / "models/heads"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
HEAD_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)

Mounted at /content/drive
Project: /content/drive/MyDrive/pa1-beyond-iid


In [4]:

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/pa1-beyond-iid")

TASK_ROOT = PROJECT_ROOT / "task1"

DATA_PATH = TASK_ROOT / "data/stl10_labeled.npz"
SPLIT_PATH = TASK_ROOT / "data/split_indices.json"

FEATURE_DIR = TASK_ROOT / "results/features"
HEAD_DIR = TASK_ROOT / "models/heads"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
HEAD_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/pa1-beyond-iid


In [5]:
with np.load(DATA_PATH, allow_pickle=False) as data:
    train_images = data["train_images"]
    train_labels = data["train_labels"]

    test_images = data["test_images"]
    test_labels = data["test_labels"]

    test_original_indices = data["test_indices"]
    CLASS_NAMES = data["class_names"].tolist()

with open(SPLIT_PATH, "r") as f:
    splits = json.load(f)

train_indices = np.array(splits["train_indices"], dtype=int)

val_indices = np.array(splits["validation_indices"], dtype=int)

assert np.array_equal(test_original_indices, np.array(splits["test_indices"]))

assert len(train_indices) == 4000
assert len(val_indices) == 1000
assert len(test_labels) == 500

print("Training:", len(train_indices))
print("Validation:", len(val_indices))
print("Evaluation:", len(test_labels))
print("Classes:", CLASS_NAMES)

Training: 4000
Validation: 1000
Evaluation: 500
Classes: ['airplane', 'bird', 'car', 'cat', 'deer', 'dog', 'horse', 'monkey', 'ship', 'truck']


In [6]:

COMMON_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor()
    ])


class CanonicalDataset(Dataset):

    def __init__(self, images, labels, indices=None):
        self.images = images
        self.labels = labels

        if indices is None:
            indices = np.arange(len(images))

        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):

        idx = int(self.indices[position])
        image_array = self.images[idx]
        image = Image.fromarray(np.transpose(image_array, (1, 2, 0))).convert("RGB")
        image = COMMON_TRANSFORM(image)
        label = int(self.labels[idx])
        return image, label


train_ds = CanonicalDataset(train_images, train_labels, train_indices)

val_ds = CanonicalDataset(train_images, train_labels, val_indices)

test_ds = CanonicalDataset(test_images, test_labels)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))
print("Sample shape:", train_ds[0][0].shape)

Train: 4000
Validation: 1000
Test: 500
Sample shape: torch.Size([3, 224, 224])


In [7]:

FEATURE_BATCH_SIZE = 32

train_loader = DataLoader(
                          train_ds,
                          batch_size=FEATURE_BATCH_SIZE,
                          shuffle=False,
                          num_workers=2,
                          pin_memory=(DEVICE.type == "cuda")
                         )

val_loader = DataLoader(
                        val_ds,
                        batch_size=FEATURE_BATCH_SIZE,
                        shuffle=False,
                        num_workers=2,
                        pin_memory=(DEVICE.type == "cuda")
                       )

test_loader = DataLoader(
                          test_ds,
                          batch_size=FEATURE_BATCH_SIZE,
                          shuffle=False,
                          num_workers=2,
                          pin_memory=(DEVICE.type == "cuda")
                        )

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Evaluation batches:", len(test_loader))

Training batches: 125
Validation batches: 32
Evaluation batches: 16


In [8]:

def load_backbone(name):

    if name == "resnet50":
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Identity()
        feature_dim = 2048

    elif name == "vit_b16":
        model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        model.heads = nn.Identity()
        feature_dim = 768

    elif name == "clip_b32":

        model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
        feature_dim = 512

    else:
        raise ValueError(f"Unknown backbone: {name}")

    for parameter in model.parameters():
        parameter.requires_grad = False

    model = model.to(DEVICE)
    model.eval()

    assert all(not p.requires_grad for p in model.parameters())

    return model, feature_dim


MODEL_NAMES = ["resnet50", "vit_b16", "clip_b32"]

FEATURE_DIMS = {"resnet50": 2048, "vit_b16": 768, "clip_b32": 512}

In [9]:

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def get_normalization(model_name):

    if model_name == "clip_b32":
        mean = CLIP_MEAN
        std = CLIP_STD
    else:
        mean = IMAGENET_MEAN
        std = IMAGENET_STD

    mean = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor(std, device=DEVICE).view(1, 3, 1, 1)

    return mean, std

In [10]:

@torch.no_grad() #! not updating the backbone

def extract_features(model, loader, model_name):

    model.eval() #! puts the backbone into evalutation mode
    mean, std = get_normalization(model_name)

    all_features = []
    all_labels = []

    for images, labels in tqdm(loader, desc=f"Extracting {model_name}"):

        images = images.to(DEVICE, non_blocking=True)
        images = (images - mean) / std

        if model_name == "clip_b32":
            features = model.encode_image(images)
            #! Performs L2 normalization
            features = F.normalize(features.float(), p=2, dim=-1)

        else:
            features = model(images)

        all_features.append(features.float().cpu())
        all_labels.append(labels.long().cpu())

    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    assert all_features.ndim == 2

    assert all_features.shape[1] == (
        FEATURE_DIMS[model_name]
    )

    assert torch.isfinite(all_features).all()

    assert len(all_features) == len(all_labels)

    return all_features, all_labels

In [11]:

@torch.no_grad()
def save_zeroshot_text_features(model):

    model.eval()

    prompts = [f"a photo of a {name}." for name in CLASS_NAMES]

    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    tokens = tokenizer(prompts).to(DEVICE)

    text_features = model.encode_text(tokens)
    text_features = F.normalize(text_features.float(),p=2,dim=-1)

    logit_scale = model.logit_scale.exp().item()

    output_path = FEATURE_DIR / "clip_zeroshot.pt"

    torch.save(
                {
                    "text_features": text_features.cpu(),
                    "logit_scale": logit_scale,
                    "prompts": prompts,
                    "class_names": CLASS_NAMES
                }, output_path
              )

    print("Saved zero-shot text embeddings.")
    print("Shape:", text_features.shape)
    print("Logit scale:", logit_scale)

In [12]:

for model_name in MODEL_NAMES:

    print(f"\n========== {model_name} ==========")

    train_path = (FEATURE_DIR / f"{model_name}_train.pt")
    val_path = (FEATURE_DIR / f"{model_name}_val.pt")

    zero_shot_path = (FEATURE_DIR / "clip_zeroshot.pt")

    already_complete = (
        train_path.exists() and
        val_path.exists() and (model_name != "clip_b32" or zero_shot_path.exists())
        )

    if already_complete:
        print("Saved features found. Skipping.")
        continue

    model, feature_dim = load_backbone(model_name)

    train_features, train_targets = extract_features(model, train_loader, model_name)
    val_features, val_targets = extract_features(model, val_loader, model_name)

    assert torch.equal(
            train_targets,
            torch.tensor(train_labels[train_indices], dtype=torch.long)
        )

    assert torch.equal(
            val_targets,
            torch.tensor(train_labels[val_indices], dtype=torch.long)
        )

    torch.save(
                {
                    "features": train_features,
                    "labels": train_targets,
                    "indices": torch.tensor(train_indices)
                }, train_path
              )

    torch.save(
                {
                    "features": val_features,
                    "labels": val_targets,
                    "indices": torch.tensor(val_indices)
                }, val_path
              )

    print("Train shape:", train_features.shape)
    print("Val shape:", val_features.shape)

    if model_name == "clip_b32":
        save_zeroshot_text_features(model)

    del model
    del train_features, val_features
    del train_targets, val_targets

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Completed:", model_name)


========== resnet50 ==========
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 140MB/s]


Extracting resnet50:   0%|          | 0/125 [00:00<?, ?it/s]

Extracting resnet50:   0%|          | 0/32 [00:00<?, ?it/s]

Train shape: torch.Size([4000, 2048])
Val shape: torch.Size([1000, 2048])
Completed: resnet50

========== vit_b16 ==========
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


...

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Extracting clip_b32:   0%|          | 0/125 [00:00<?, ?it/s]

Extracting clip_b32:   0%|          | 0/32 [00:00<?, ?it/s]

Train shape: torch.Size([4000, 512])
Val shape: torch.Size([1000, 512])
Saved zero-shot text embeddings.
Shape: torch.Size([10, 512])
Logit scale: 100.0
Completed: clip_b32


In [13]:

def train_linear_head(model_name):
    print(f"\nTraining head for {model_name}")

    train_data = torch.load(FEATURE_DIR / f"{model_name}_train.pt", map_location="cpu", weights_only=True)
    val_data = torch.load(FEATURE_DIR / f"{model_name}_val.pt", map_location="cpu", weights_only=True)

    X_train = train_data["features"].float()
    y_train = train_data["labels"].long()

    X_val = val_data["features"].float()
    y_val = val_data["labels"].long()

    #! Reset the seed for each independent comparison.
    torch.manual_seed(SEED)

    head = nn.Linear(X_train.shape[1], len(CLASS_NAMES))

    optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(SEED)

    loader = DataLoader(
                          TensorDataset(X_train, y_train),
                          batch_size=128,
                          shuffle=True,
                          generator=generator
                       )
    best_accuracy = -1.0
    best_epoch = 0
    best_state = None

    patience = 5
    epochs_without_improvement = 0

    history = []

    for epoch in range(1, 51):
        head.train()

        total_loss = 0.0
        total_examples = 0

        for features, labels in loader:
            optimizer.zero_grad(set_to_none=True)
            logits = head(features)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += (loss.item() * len(labels))
            total_examples += len(labels)

        train_loss = total_loss / total_examples

        # Validation: no parameter updates.
        head.eval()

        with torch.no_grad():
            val_logits = head(X_val)
            val_predictions = val_logits.argmax(dim=1)
            val_accuracy = ((val_predictions == y_val).float().mean().item())
            val_loss = F.cross_entropy(val_logits, y_val).item()

        history.append({
                          "epoch": epoch,
                          "train_loss": train_loss,
                          "val_loss": val_loss,
                          "val_accuracy": val_accuracy
                      })

        print(
              f"Epoch {epoch:02d} | "
              f"Train loss: {train_loss:.4f} | "
              f"Val loss: {val_loss:.4f} | "
              f"Val accuracy: {val_accuracy:.4f}"
             )

        if val_accuracy > best_accuracy + 1e-12:
            best_accuracy = val_accuracy
            best_epoch = epoch
            best_state = copy.deepcopy(head.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print("Early stopping triggered.")
            break

    # Restore the best validation checkpoint.
    head.load_state_dict(best_state)

    checkpoint_path = (HEAD_DIR / f"{model_name}_head.pt")

    torch.save(
        {
            "model_name": model_name,
            "head_state_dict": head.state_dict(),
            "in_features": X_train.shape[1],
            "num_classes": len(CLASS_NAMES),
            "best_epoch": best_epoch,
            "best_val_accuracy": best_accuracy,
            "seed": SEED
        }, checkpoint_path
    )

    history_path = (HEAD_DIR / f"{model_name}_history.json")

    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    print("\nBest epoch:", best_epoch)
    print("Best validation accuracy:", best_accuracy)
    print("Checkpoint saved:", checkpoint_path)

    return {
            "model": model_name,
            "best_epoch": best_epoch,
            "best_val_accuracy": best_accuracy
           }

In [14]:

training_results = []

for model_name in MODEL_NAMES:
    result = train_linear_head(model_name)
    training_results.append(result)

print("\n========== SUMMARY ==========")

for result in training_results:
    print(result["model"], "| Best epoch:", result["best_epoch"], "| Val accuracy:", f'{result["best_val_accuracy"]:.4f}')


Training head for resnet50
Epoch 01 | Train loss: 1.4758 | Val loss: 0.8454 | Val accuracy: 0.9530
Epoch 02 | Train loss: 0.5702 | Val loss: 0.4216 | Val accuracy: 0.9620
Epoch 03 | Train loss: 0.3102 | Val loss: 0.2812 | Val accuracy: 0.9620
Epoch 04 | Train loss: 0.2105 | Val loss: 0.2173 | Val accuracy: 0.9670
Epoch 05 | Train loss: 0.1581 | Val loss: 0.1810 | Val accuracy: 0.9710
Epoch 06 | Train loss: 0.1261 | Val loss: 0.1570 | Val accuracy: 0.9710
Epoch 07 | Train loss: 0.1041 | Val loss: 0.1413 | Val accuracy: 0.9720
Epoch 08 | Train loss: 0.0881 | Val loss: 0.1297 | Val accuracy: 0.9710
Epoch 09 | Train loss: 0.0758 | Val loss: 0.1208 | Val accuracy: 0.9720
Epoch 10 | Train loss: 0.0665 | Val loss: 0.1140 | Val accuracy: 0.9730
Epoch 11 | Train loss: 0.0586 | Val loss: 0.1082 | Val accuracy: 0.9730
Epoch 12 | Train loss: 0.0523 | Val loss: 0.1042 | Val accuracy: 0.9730
Epoch 13 | Train loss: 0.0471 | Val loss: 0.1011 | Val accuracy: 0.9720
Epoch 14 | Train loss: 0.0426 | Val 

In [15]:
expected_labels = torch.tensor(test_labels, dtype=torch.long)
expected_indices = torch.tensor(test_original_indices, dtype=torch.long)

for model_name in MODEL_NAMES:
    test_path = FEATURE_DIR / f"{model_name}_test.pt"

    if test_path.exists():
        saved = torch.load(test_path, map_location="cpu", weights_only=True)
        assert saved["features"].shape == (500, FEATURE_DIMS[model_name])
        assert torch.equal(saved["labels"], expected_labels)
        assert torch.equal(saved["indices"], expected_indices)
        print(f"{model_name}: Test features already saved.")
        continue

    print(f"\nExtracting test features: {model_name}")

    model, feature_dim = load_backbone(model_name)
    test_features, test_targets = extract_features(model, test_loader, model_name)

    assert test_features.shape == (500, feature_dim)
    assert torch.equal(test_targets, expected_labels)

    torch.save(
        {
            "features": test_features.cpu(),
            "labels": test_targets.cpu(),
            "indices": expected_indices,
            "seed": SEED
        }, test_path
    )

    print(f"Saved: {test_path}")
    print(f"Shape: {test_features.shape}")

    del model, test_features, test_targets
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nDONE! All three test-feature files saved to Google Drive.")


Extracting test features: resnet50


Extracting resnet50:   0%|          | 0/16 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/pa1-beyond-iid/task1/results/features/resnet50_test.pt
Shape: torch.Size([500, 2048])

Extracting test features: vit_b16


Extracting vit_b16:   0%|          | 0/16 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/pa1-beyond-iid/task1/results/features/vit_b16_test.pt
Shape: torch.Size([500, 768])



Extracting test features: clip_b32


Extracting clip_b32:   0%|          | 0/16 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/pa1-beyond-iid/task1/results/features/clip_b32_test.pt
Shape: torch.Size([500, 512])

DONE! All three test-feature files saved to Google Drive.
